In [9]:
!pip install faiss-cpu

In [10]:
import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [11]:
documents = pd.read_csv("/content/drive/MyDrive/immverse.ai/cleaned_dataset.csv")

documents.head()

,unique_key,sanskrit_data,translation,document
0,1,धृतराष्ट्र उवाचधर्मक्षेत्रे कुरुक्षेत्रे समवेत...,Bhagavad-gītā is the widely read theistic scie...,Verse ID: 1\n\nSanskrit:\nधृतराष्ट्र उवाचधर्मक...
1,2,सञ्जय उवाचदृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्य...,Dhṛtarāṣṭra was blind from birth. Unfortunatel...,Verse ID: 2\n\nSanskrit:\nसञ्जय उवाचदृष्ट्वा त...
2,3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् ।व्...,"Duryodhana, a great diplomat, wanted to point ...",Verse ID: 3\n\nSanskrit:\nपश्यैतां पाण्डुपुत्र...
3,4,अत्र श‍ूरा महेष्वासा भीमार्जुनसमा युधि ।युयुधा...,Even though Dhṛṣṭadyumna was not a very import...,Verse ID: 4\n\nSanskrit:\nअत्र श‍ूरा महेष्वासा...
4,5,धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् ।पुरु...,"There are also great heroic, powerful fighters...",Verse ID: 5\n\nSanskrit:\nधृष्टकेतुश्चेकितानः ...


In [12]:
MODEL_PATH = "/content/drive/MyDrive/immverse.ai/models"

model = SentenceTransformer(MODEL_PATH)

print("Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model Loaded


In [13]:
docs = documents["document"].tolist()

len(docs)

657

In [14]:
embeddings = model.encode(

    docs,

    convert_to_numpy=True,

    show_progress_bar=True

)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [15]:
embeddings.shape

(657, 384)

In [16]:
faiss.normalize_L2(embeddings)

In [17]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(index.ntotal)

657


In [18]:
faiss.write_index(

    index,

    "/content/drive/MyDrive/immverse.ai/faiss_index.bin"

)

print("FAISS index saved.")

FAISS index saved.


In [26]:
faiss.write_index(index, "/content/drive/MyDrive/immverse.ai/faiss.index")

In [19]:
with open("/content/drive/MyDrive/immverse.ai/documents.pkl","wb") as f:

    pickle.dump(docs,f)

print("Documents saved.")

Documents saved.


In [20]:
metadata = documents[
    [
        "unique_key",
        "sanskrit_data",
        "translation"
    ]
]

metadata.to_pickle("/content/drive/MyDrive/immverse.ai/metadata.pkl")

print("Metadata saved.")

Metadata saved.


In [21]:
query = "What does Krishna say about Karma?"

In [22]:
query_embedding = model.encode(

    [query],

    convert_to_numpy=True

)

faiss.normalize_L2(query_embedding)

In [23]:
k = 5

scores, indices = index.search(

    query_embedding,

    k

)

In [24]:
for rank, idx in enumerate(indices[0]):

    print("="*80)

    print("Rank:", rank+1)

    print("Score:", scores[0][rank])

    print()

    print(metadata.iloc[idx]["translation"][:500])

    print()

    print(metadata.iloc[idx]["sanskrit_data"])

Rank: 1
Score: 0.27421165

A person in Kṛṣṇa consciousness certainly sees Lord Kṛṣṇa everywhere, and he sees everything in Kṛṣṇa. Such a person may appear to see all separate manifestations of the material nature, but in each and every instance he is conscious of Kṛṣṇa, knowing that everything is a manifestation of Kṛṣṇa’s energy. Nothing can exist without Kṛṣṇa, and Kṛṣṇa is the Lord of everything – this is the basic principle of Kṛṣṇa consciousness. Kṛṣṇa consciousness is the development of love of Kṛṣṇa – a position tr

यो मां पश्यति सर्वत्र सर्वं च मयि पश्यति ।तस्याहं न प्रणश्यामि स च मे न प्रणश्यति ॥ ३० ॥
Rank: 2
Score: 0.24840239

A person acting in Kṛṣṇa consciousness is naturally free from the bonds of karma. His activities are all performed for Kṛṣṇa; therefore he does not enjoy or suffer any of the effects of work. Consequently he is intelligent in human society, even though he is engaged in all sorts of activities for Kṛṣṇa. Akarma means without reaction to work. The imperso

In [25]:
print("FAISS Index Built Successfully")

print("Documents Indexed :", index.ntotal)

print("Embedding Dimension :", dimension)

FAISS Index Built Successfully
Documents Indexed : 657
Embedding Dimension : 384
